<a href="https://colab.research.google.com/github/flahbocchino/cardioia-fase1-tabagismo-vape/blob/main/cardioia_fase1_vc_100_imagens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

base = Path("/content/cardioia_vc_imagens")
folders = ["ecg", "rx_torax", "angiografia", "ecocardiograma"]

for f in folders:
    (base / f).mkdir(parents=True, exist_ok=True)

print("Pasta base:", base)
for f in folders:
    print("-", (base / f))


Pasta base: /content/cardioia_vc_imagens
- /content/cardioia_vc_imagens/ecg
- /content/cardioia_vc_imagens/rx_torax
- /content/cardioia_vc_imagens/angiografia
- /content/cardioia_vc_imagens/ecocardiograma


In [2]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# ---------- 1) ECG (traçado) ----------
def ecg_like_signal(seconds=10, fs=250, hr=70, noise=0.02):
    n = int(seconds * fs)
    t = np.linspace(0, seconds, n)
    beat_interval = 60 / hr
    peaks = np.arange(0.7, seconds, beat_interval)

    sig = 0.02*np.sin(2*np.pi*0.33*t)  # baseline wander leve

    for p in peaks:
        sig += np.exp(-((t-p)/0.012)**2) * 1.0      # R
        sig -= np.exp(-((t-(p-0.02))/0.02)**2)*0.12 # Q
        sig -= np.exp(-((t-(p+0.02))/0.02)**2)*0.20 # S
        sig += np.exp(-((t-(p+0.25))/0.06)**2)*0.18 # T

    sig += rng.normal(0, noise, size=n)
    return t, sig

def save_ecg(path, idx):
    hr = int(rng.integers(55, 115))
    noise = float(rng.uniform(0.01, 0.06))
    t, sig = ecg_like_signal(hr=hr, noise=noise)

    plt.figure(figsize=(10, 3))
    plt.plot(t, sig)
    plt.title(f"ECG - amostra {idx:02d} (HR≈{hr} bpm)")
    plt.xlabel("Tempo (s)")
    plt.ylabel("Amplitude (a.u.)")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()

# ---------- 2) RX Tórax (estilizado) ----------
def synthetic_xray(h=512, w=512):
    img = rng.normal(0.5, 0.08, (h, w))

    yy, xx = np.mgrid[0:h, 0:w]
    cx, cy = w*0.5, h*0.52

    # “Pulmões” (2 elipses mais escuras)
    left_lung  = ((xx-(cx-90))**2)/(120**2) + ((yy-(cy))**2)/(170**2) < 1
    right_lung = ((xx-(cx+90))**2)/(120**2) + ((yy-(cy))**2)/(170**2) < 1
    img[left_lung]  -= 0.18
    img[right_lung] -= 0.18

    # “Coração” (elipse mais clara no centro)
    heart = ((xx-cx)**2)/(95**2) + ((yy-(cy+30))**2)/(110**2) < 1
    img[heart] += 0.15

    # “Costelas” (linhas suaves)
    for k in range(6):
        y = int(h*0.18 + k*55 + rng.integers(-4, 5))
        img[max(0,y-2):min(h,y+2), :] += 0.03

    img = np.clip(img, 0, 1)
    return img

def save_xray(path, idx):
    img = synthetic_xray()
    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap="gray")
    plt.title(f"RX Tórax - amostra {idx:02d}")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()

# ---------- 3) Angiografia (estilizado) ----------
def synthetic_angiogram(h=512, w=512):
    img = rng.normal(0.15, 0.05, (h, w))  # fundo escuro
    yy, xx = np.mgrid[0:h, 0:w]

    # vasos como “caminhos” brilhantes
    x = w*0.5 + rng.normal(0, 5)
    y = h*0.15
    for _ in range(900):
        x += rng.normal(0, 2.2)
        y += abs(rng.normal(0, 1.6))
        if 5 < x < w-5 and 5 < y < h-5:
            img[int(y)-1:int(y)+2, int(x)-1:int(x)+2] += 0.35

        # bifurcação ocasional
        if rng.random() < 0.004 and y < h*0.85:
            xb, yb = x, y
            for _ in range(200):
                xb += rng.normal(0, 2.8)
                yb += abs(rng.normal(0, 1.6))
                if 5 < xb < w-5 and 5 < yb < h-5:
                    img[int(yb)-1:int(yb)+2, int(xb)-1:int(xb)+2] += 0.28

    img = np.clip(img, 0, 1)
    return img

def save_angiogram(path, idx):
    img = synthetic_angiogram()
    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap="gray")
    plt.title(f"Angiografia - amostra {idx:02d}")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()

# ---------- 4) Ecocardiograma (estilizado) ----------
def synthetic_echo(h=512, w=512):
    img = rng.normal(0.35, 0.12, (h, w))

    yy, xx = np.mgrid[0:h, 0:w]
    cx, cy = w*0.52, h*0.55

    # “cavidade” escura (elipse)
    cavity = ((xx-cx)**2)/(140**2) + ((yy-cy)**2)/(110**2) < 1
    img[cavity] -= 0.20

    # “parede” mais clara em volta (anel)
    wall_outer = ((xx-cx)**2)/(160**2) + ((yy-cy)**2)/(130**2) < 1
    wall = wall_outer & (~cavity)
    img[wall] += 0.10

    # granulação típica
    img = np.clip(img, 0, 1)
    return img

def save_echo(path, idx):
    img = synthetic_echo()
    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap="gray")
    plt.title(f"Ecocardiograma - amostra {idx:02d}")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()

# ---------- Gerar 25 de cada ----------
from pathlib import Path
base = Path("/content/cardioia_vc_imagens")

for i in range(1, 26):
    save_ecg(base / "ecg" / f"ecg_{i:02d}.png", i)
    save_xray(base / "rx_torax" / f"rx_{i:02d}.png", i)
    save_angiogram(base / "angiografia" / f"angio_{i:02d}.png", i)
    save_echo(base / "ecocardiograma" / f"eco_{i:02d}.png", i)

print("OK! Imagens geradas em:", base)


OK! Imagens geradas em: /content/cardioia_vc_imagens


In [4]:
from pathlib import Path

base = Path("/content/cardioia_vc_imagens")
imgs = list(base.rglob("*.png"))

print("Total de imagens:", len(imgs))
print("Alguns exemplos:")
for p in imgs[:10]:
    print(p)


Total de imagens: 100
Alguns exemplos:
/content/cardioia_vc_imagens/angiografia/angio_19.png
/content/cardioia_vc_imagens/angiografia/angio_14.png
/content/cardioia_vc_imagens/angiografia/angio_09.png
/content/cardioia_vc_imagens/angiografia/angio_24.png
/content/cardioia_vc_imagens/angiografia/angio_05.png
/content/cardioia_vc_imagens/angiografia/angio_06.png
/content/cardioia_vc_imagens/angiografia/angio_01.png
/content/cardioia_vc_imagens/angiografia/angio_17.png
/content/cardioia_vc_imagens/angiografia/angio_10.png
/content/cardioia_vc_imagens/angiografia/angio_02.png


In [6]:
%%bash
cd /content
zip -r cardioia_vc_imagens_100.zip cardioia_vc_imagens
ls -lh cardioia_vc_imagens_100.zip


  adding: cardioia_vc_imagens/ (stored 0%)
  adding: cardioia_vc_imagens/angiografia/ (stored 0%)
  adding: cardioia_vc_imagens/angiografia/angio_19.png (deflated 3%)
  adding: cardioia_vc_imagens/angiografia/angio_14.png (deflated 2%)
  adding: cardioia_vc_imagens/angiografia/angio_09.png (deflated 3%)
  adding: cardioia_vc_imagens/angiografia/angio_24.png (deflated 3%)
  adding: cardioia_vc_imagens/angiografia/angio_05.png (deflated 3%)
  adding: cardioia_vc_imagens/angiografia/angio_06.png (deflated 3%)
  adding: cardioia_vc_imagens/angiografia/angio_01.png (deflated 2%)
  adding: cardioia_vc_imagens/angiografia/angio_17.png (deflated 3%)
  adding: cardioia_vc_imagens/angiografia/angio_10.png (deflated 3%)
  adding: cardioia_vc_imagens/angiografia/angio_02.png (deflated 3%)
  adding: cardioia_vc_imagens/angiografia/angio_20.png (deflated 3%)
  adding: cardioia_vc_imagens/angiografia/angio_21.png (deflated 2%)
  adding: cardioia_vc_imagens/angiografia/angio_16.png (deflated 3%)
  add

In [7]:
from google.colab import files
files.download("/content/cardioia_vc_imagens_100.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>